# TRELLIS image -> 3D building (Colab GPU)

Backend **generative**: biến một ảnh tòa nhà (ảnh thật/render) thành mesh GLB realistic
bằng [TRELLIS](https://github.com/microsoft/TRELLIS), rồi lắp thành `SceneBundle` (describer
tiếng Việt chạy CPU). Scene generative KHÔNG có drill phòng (Explorer fallback model-viewer).

**Yêu cầu:** Runtime = GPU (Menu Runtime -> Change runtime type -> T4/L4/A100, >=6GB VRAM).

Lưu ý: cài đặt TRELLIS khá nặng và nhạy phiên bản; nếu bước setup lỗi, theo README của
TRELLIS để chỉnh cờ. Backend procedural (CPU, build.ipynb) vẫn là đường mặc định chạy mọi máy.

In [ ]:
!nvidia-smi -L  # xác nhận có GPU

## Bước 1 - Lấy code (repo của bạn + TRELLIS)

In [ ]:
REPO_URL = "https://github.com/hoaianthai345/3D_Building_Builder.git"
%cd /content
!rm -rf repo && git clone --depth 1 $REPO_URL repo
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS

## Bước 2 - Cài TRELLIS (vài phút) + deps lắp bundle
Theo README TRELLIS; có thể cần chỉnh cờ tùy môi trường Colab hiện tại.

In [ ]:
%cd /content/TRELLIS
!. ./setup.sh --basic --xformers --spconv --mipgaussian --nvdiffrast
%cd /content
!pip install -q -r repo/requirements.txt

## Bước 3 - Tải lên ảnh tòa nhà
Ảnh chính diện/phối cảnh rõ ràng cho kết quả tốt nhất.

In [ ]:
from google.colab import files
up = files.upload()
image_path = '/content/' + list(up)[0]
print('image:', image_path)

## Bước 4 - Sinh mesh + lắp SceneBundle

In [ ]:
import os, sys
os.environ.setdefault('ATTN_BACKEND', 'xformers')
os.environ.setdefault('SPCONV_ALGO', 'native')
sys.path.insert(0, '/content/TRELLIS')
sys.path.insert(0, '/content/repo')
os.chdir('/content/repo')

from builder.pipeline import generate_from_image
from builder.schemas import GenerateRequest, SpaceType

bundle = generate_from_image(
    GenerateRequest(
        project_name='Tòa nhà từ ảnh',
        space_type=SpaceType.office,
        description='Mô tả ngắn cho tòa nhà trong ảnh',
        target_audience='Khách hàng mục tiêu',
    ),
    image_path=image_path,
    out_dir='frontend/public/artifacts',
)
print('id    :', bundle.id, '| backend:', bundle.model.backend)
print('model :', bundle.model.glb, bundle.model.tri_count, 'tris,', bundle.model.size_kb, 'KB')
print('title :', bundle.describer.title)

## Bước 5 - Xem GLB inline

In [ ]:
import base64
from IPython.display import HTML
glb = f"frontend/public/artifacts/{bundle.model.glb}"
b64 = base64.b64encode(open(glb, 'rb').read()).decode()
HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{b64}" camera-controls auto-rotate
  shadow-intensity="0.7" exposure="1.0" style="width:100%;height:460px;background:#f0eee6;border-radius:16px"></model-viewer>
''')

## Bước 6 - Tải artifact về
Copy vào `frontend/public/artifacts/` của repo để demo đọc (scene generative hiện trong model-viewer).

In [ ]:
import shutil
shutil.make_archive('scene-generative', 'zip', 'frontend/public/artifacts')
from google.colab import files
files.download('scene-generative.zip')